# 02 — Unsloth + QLoRA 파인튜닝

**목적:** `train.jsonl`로 Qwen2.5-1.5B-Instruct를 기술 추출 태스크에 파인튜닝.

**런타임:** T4 GPU (Colab 무료) 필수  
**예상 학습 시간:** 550샘플 × 3에포크 ≈ 25분  
**체크포인트:** Google Drive에 저장 (세션 종료 후에도 유지)

---
### 선행 조건
- `01_generate_dataset.ipynb` 실행 완료 (`train.jsonl` / `test.jsonl` 생성)
- Colab 왼쪽 사이드바 🔑 **Secrets** 에 `HF_TOKEN` 등록 (HF Hub 업로드용)
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

In [ ]:
# GPU 확인 — T4가 아니면 진행하지 마세요
import subprocess

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, check=True,
    )
    print('GPU:', result.stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print('[경고] nvidia-smi 없음 — Mac/CPU 환경입니다.')
    print('  Colab에서 실행하려면: 런타임 → 런타임 유형 변경 → T4 GPU 선택')

In [ ]:
# Unsloth + TRL 설치 (Colab 전용)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q unsloth trl transformers datasets peft accelerate bitsandbytes huggingface_hub
    # xformers는 선택사항 — 없어도 동작
    !pip install -q xformers --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || true
print('설치 완료')

In [ ]:
import os
import json
from pathlib import Path

import torch

# ── 경로 및 하이퍼파라미터 설정 ────────────────────────────────
if IN_COLAB:
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    HF_TOKEN   = userdata.get('HF_TOKEN', '')
    HF_REPO_ID = userdata.get('HF_REPO_ID', '')  # e.g. 'your_username/qwen2.5-1.5b-job-skill-extractor'
    DATASET_DIR = Path('/content/drive/MyDrive/job-skill-ft/dataset')
    OUTPUT_DIR  = Path('/content/drive/MyDrive/job-skill-ft/checkpoints')
else:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN   = os.getenv('HF_TOKEN', '')
    HF_REPO_ID = os.getenv('HF_REPO_ID', '')
    DATASET_DIR = Path('finetune/dataset')
    OUTPUT_DIR  = Path('finetune/checkpoints')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 학습 하이퍼파라미터
MODEL_NAME   = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_SEQ_LEN  = 1024
LORA_RANK    = 16
LORA_ALPHA   = 32
LEARNING_RATE = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4   # Effective batch = 16
NUM_EPOCHS   = 3
SAVE_STEPS   = 50
WARMUP_RATIO = 0.1

print(f'모델   : {MODEL_NAME}')
print(f'데이터 : {DATASET_DIR}')
print(f'출력   : {OUTPUT_DIR}')
print(f'HF Hub : {HF_REPO_ID or "(설정 안됨 — 업로드 건너뜀)"}')
print(f'CUDA   : {torch.cuda.is_available()}')

In [ ]:
# ── 모델 + 토크나이저 로드 (Unsloth 4-bit) ────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # None → auto (T4는 float16)
    load_in_4bit=True,
)
print(f'로드 완료 | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── LoRA 어댑터 추가 ────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # VRAM 절약
    random_state=42,
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터    : {total_params/1e6:.1f}M')
print(f'학습 파라미터    : {trainable/1e6:.2f}M ({trainable/total_params*100:.2f}%)')

In [ ]:
# ── 데이터셋 로드 + ChatML 포맷팅 ───────────────────────────────
from datasets import Dataset

SYSTEM_MSG = (
    'You are a technical skill extractor. '
    'Extract skills from job postings and return valid JSON only. '
    'No explanation or markdown fences.'
)

def load_jsonl(path: Path) -> list[dict]:
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def format_to_chatml(example: dict) -> dict:
    """Alpaca 형식(instruction/input/output) → Qwen2.5 ChatML 형식 변환."""
    messages = [
        {'role': 'system',    'content': SYSTEM_MSG},
        {'role': 'user',      'content': f"{example['instruction']}\n\n{example['input']}"},
        {'role': 'assistant', 'content': example['output']},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # 학습 시에는 EOS 토큰까지 포함
    )
    return {'text': text}


train_raw = load_jsonl(DATASET_DIR / 'train.jsonl')
eval_raw  = load_jsonl(DATASET_DIR / 'test.jsonl')

train_ds = Dataset.from_list(train_raw).map(format_to_chatml, remove_columns=['instruction', 'input', 'output'])
eval_ds  = Dataset.from_list(eval_raw).map(format_to_chatml,  remove_columns=['instruction', 'input', 'output'])

print(f'Train: {len(train_ds)}개 | Eval: {len(eval_ds)}개')

# 토큰 길이 분포 확인
lengths = [len(tokenizer(ex['text'])['input_ids']) for ex in train_ds.select(range(min(50, len(train_ds))))]
print(f'샘플 토큰 길이 (처음 50개) — min: {min(lengths)}, avg: {sum(lengths)//len(lengths)}, max: {max(lengths)}')

In [ ]:
# ── SFTTrainer 설정 + 학습 ──────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    weight_decay=0.01,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    seed=42,
    report_to='none',  # wandb 비활성화
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print('Trainer 준비 완료. 학습 시작...')
trainer_stats = trainer.train()
print(f'학습 완료! 소요 시간: {trainer_stats.metrics["train_runtime"]:.0f}초')

In [ ]:
# ── 추론 테스트 ────────────────────────────────────────────────
import json as _json

FastLanguageModel.for_inference(model)

TEST_JOB = {
    'title': 'Senior LLM Application Engineer',
    'description': (
        'Required: Python, LangGraph for multi-agent systems, vector databases (Chroma or Weaviate), '
        'REST API design with FastAPI, Docker. '
        'Preferred: Neo4j, RAGAS evaluation framework, Langfuse tracing, AWS or GCP.'
    ),
}

INSTRUCTION = (
    'Extract technical skills from the job posting below. '
    'Return valid JSON only with this exact schema — no markdown fences, no commentary:\n'
    '{"job_title": "string", "skills": [{"raw": "string", '
    '"category": "language|framework|database|cloud|tool|concept", '
    '"importance": "required|preferred"}]}'
)

messages = [
    {'role': 'system',   'content': SYSTEM_MSG},
    {'role': 'user',     'content': f"{INSTRUCTION}\n\nTitle: {TEST_JOB['title']}\n\nDescription:\n{TEST_JOB['description']}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.0,
        do_sample=False,
    )
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

print('=== 모델 출력 ===')
print(response)
try:
    parsed = _json.loads(response.strip())
    print(f'\nJSON 파싱 성공 — 추출 기술 {len(parsed.get("skills", []))}개')
    for s in parsed.get('skills', []):
        mark = '✅' if s['importance'] == 'required' else '⬜'
        print(f'  {mark} [{s["category"]:12}] {s["raw"]}')
except Exception as e:
    print(f'JSON 파싱 실패: {e}')

In [ ]:
# ── HF Hub 업로드 ─────────────────────────────────────────────
if not HF_TOKEN:
    print('[skip] HF_TOKEN이 없어 업로드를 건너뜁니다.')
elif not HF_REPO_ID:
    print('[skip] HF_REPO_ID가 없어 업로드를 건너뜁니다.')
    print('  Secrets에 HF_REPO_ID = "your_username/qwen2.5-1.5b-job-skill-extractor" 를 추가하세요.')
else:
    print(f'HF Hub 업로드 시작: {HF_REPO_ID}')

    # LoRA 어댑터만 업로드 (~120MB)
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

    print(f'\n업로드 완료: https://huggingface.co/{HF_REPO_ID}')
    print('다음 단계: 03_evaluate.ipynb에서 성능 측정 후 모델 카드를 업데이트하세요.')

In [ ]:
# ── 로컬 저장 (Drive 백업용) ────────────────────────────────────
LOCAL_SAVE = OUTPUT_DIR / 'best_adapter'
model.save_pretrained(str(LOCAL_SAVE))
tokenizer.save_pretrained(str(LOCAL_SAVE))
print(f'로컬 저장 완료: {LOCAL_SAVE}')

# 저장된 파일 목록
for f in sorted(LOCAL_SAVE.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:40} {size_mb:.1f} MB')